In [1]:
%%capture
!pip install --upgrade unsloth
!pip install transformers peft datasets trl -q

In [2]:
import os, sys, json, time, re, warnings, logging, torch
from pathlib import Path

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')

GPU: NVIDIA L4
VRAM: 22.0 GB
PyTorch: 2.10.0+cu128


In [ ]:
# Mount Google Drive — MUST run before any code creates paths under
# /content/drive, otherwise drive.mount() fails ('Mountpoint must not
# already contain files') or a stray local dir tricks a naive exists()
# check into skipping the real mount, silently writing checkpoints to
# ephemeral Colab storage instead of Drive.
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [3]:
# ====== SHARED CONFIG — chỉnh ở đây ======
CONFIG = {
    # Kaggle API
    'kaggle_username': 'thnhcngl',

    # Kaggle datasets
    'ds_sft_ckpt': 'thnhcngl/dvsktt-sft-best-checkpoint',
    'ds_sft_data': 'thnhcngl/dvsktt-ner-sft',
    'ds_han_data': 'thnhcngl/dvsktt-han-pretrain',

    # Local paths
    'data_dir':    '/content/data',
    'work_dir':    '/content/drive/MyDrive/dvsktt_ner',
    'ckpt_dir':    '/content/drive/MyDrive/dvsktt_ner/checkpoints',
    'result_dir':  '/content/drive/MyDrive/dvsktt_ner/results',
    'log_dir':     '/content/drive/MyDrive/dvsktt_ner/logs',

    # Model
    'base_model':   'unsloth/qwen2.5-7b-unsloth-bnb-4bit',
    'max_seq_len':  512,
    'lora_rank':    16,
    'lora_alpha':   32,
    'lora_dropout': 0.05,

    # Pretrain
    'pretrain_lr':     5e-5,
    'pretrain_epochs': 1,
    'pretrain_batch':  2,
    'pretrain_sample': 5000,
    'pretrain_grad_accum': 4,

    # SFT
    'sft_lr':          5e-5,
    'sft_epochs':      3,
    'sft_batch':       1,
    'sft_grad_accum':  4,

    # Evaluate
    'eval_batch':      4,
    'max_new_tokens':  900,
    'save_every':      50,

    # Resume
    'resume_from_step': 0,  # đặt số step để resume, 0 = train từ đầu
}

# Tạo thư mục
for d in ['data_dir','work_dir','ckpt_dir','result_dir','log_dir']:
    os.makedirs(CONFIG[d], exist_ok=True)

print('Config loaded. Dirs created.')
print(f"Work dir: {CONFIG['work_dir']}")

Config loaded. Dirs created.
Work dir: /content/drive/MyDrive/dvsktt_ner


In [ ]:
# Setup Kaggle API — credentials are NEVER hardcoded here.
# Set them as Colab Secrets (key icon in the left sidebar) named
# KAGGLE_USERNAME / KAGGLE_KEY, or export them as env vars if running elsewhere.
import os

try:
    from google.colab import userdata
    os.environ.setdefault('KAGGLE_USERNAME', userdata.get('KAGGLE_USERNAME'))
    os.environ.setdefault('KAGGLE_KEY', userdata.get('KAGGLE_KEY'))
except Exception:
    pass

if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
    raise RuntimeError(
        'Missing Kaggle credentials. Set KAGGLE_USERNAME/KAGGLE_KEY as Colab Secrets '
        '(key icon in sidebar) or environment variables before running this cell.'
    )

print('Kaggle API configured via environment variables.')
!kaggle --version

In [ ]:
import subprocess

def download_dataset(ds_name, data_dir, max_retries=4, retry_delay=15):
    name = ds_name.split('/')[-1]
    dest = f'{data_dir}/{name}'

    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'Already exists: {dest}')
        return dest

    os.makedirs(dest, exist_ok=True)

    # Kaggle API rate-limits rapid successive calls (403 Forbidden on
    # GetDatasetMetadata) — retry with backoff instead of failing fast.
    r = None
    for attempt in range(1, max_retries + 1):
        print(f'Downloading {ds_name} (attempt {attempt}/{max_retries})...')
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', ds_name, '-p', dest, '--unzip'],
            capture_output=True, text=True
        )
        print(r.stdout[:200])
        if r.returncode == 0:
            break
        print(f'ERROR: {r.stderr[:200]}')
        if attempt < max_retries:
            print(f'Retrying in {retry_delay}s...')
            time.sleep(retry_delay)
    if r is None or r.returncode != 0:
        return None

    print(f'Done: {dest}')
    for f in os.listdir(dest):
        print(f'  {f}')
    return dest

# dvsktt-han-pretrain và dvsktt-ner-sft là dataset Private trên Kaggle và
# liên tục bị 403 Forbidden qua API (không phải rate-limit — đã retry vẫn fail)
# nhưng cả 2 đã có sẵn ngay trong repo này (data/raw/), nên đọc thẳng từ đó,
# khỏi cần gọi Kaggle API cho 2 dataset này. Chỉ checkpoint (quá lớn cho git)
# mới cần tải qua Kaggle.
sft_ckpt_path = download_dataset(CONFIG['ds_sft_ckpt'], CONFIG['data_dir'])
REPO_DATA_DIR = '/content/repo/data/raw'
han_data_path = f'{REPO_DATA_DIR}/han_pretrain'
sft_data_path = f'{REPO_DATA_DIR}/ner_sft'


In [10]:
# ====== Logger — ghi log ra file + console ======
class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        self.start    = time.time()
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, 'a') as f:
            f.write(f'\n===== Session started: {time.strftime("%Y-%m-%d %H:%M:%S")} =====\n')

    def log(self, msg, also_print=True):
        elapsed = time.time() - self.start
        line    = f'[{elapsed:>8.1f}s] {msg}'
        with open(self.log_path, 'a') as f:
            f.write(line + '\n')
        if also_print:
            print(line)

    def save_state(self, state, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        with open(path, 'w') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        self.log(f'State saved: {path}')
        return path

    def load_state(self, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        if os.path.exists(path):
            with open(path) as f:
                state = json.load(f)
            self.log(f'State loaded: {path}')
            return state
        return None

print('Logger ready.')

Logger ready.


## Load Model

In [11]:
from unsloth import FastLanguageModel
from peft import PeftModel
from collections import defaultdict, Counter

logger = Logger(f"{CONFIG['log_dir']}/evaluate.log")

# Load model — ưu tiên sft_best local, fallback Kaggle checkpoint
local_best = f"{CONFIG['ckpt_dir']}/sft_best"
if os.path.exists(local_best):
    ckpt_path = local_best
    logger.log(f'Using local sft_best: {ckpt_path}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = ckpt_path,
        max_seq_length = CONFIG['max_seq_len'],
        load_in_4bit   = True,
        dtype          = None,
    )
else:
    logger.log(f'Using Kaggle checkpoint: {sft_ckpt_path}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = CONFIG['base_model'],
        max_seq_length = CONFIG['max_seq_len'],
        load_in_4bit   = True,
        dtype          = None,
    )
    model = PeftModel.from_pretrained(model, sft_ckpt_path)

FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'  # required for correct batched generation on a decoder-only model
logger.log('Model loaded for inference!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
[     0.0s] Using Kaggle checkpoint: /content/data/dvsktt-sft-best-checkpoint
==((====))==  Unsloth 2026.7.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[    56.6s] Model loaded for inference!


## Load Test Data

In [12]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f]

test_data = load_jsonl(f'{sft_data_path}/test.jsonl')
logger.log(f'Test records: {len(test_data):,}')

# Resume support
eval_state   = logger.load_state('eval_state')
START_IDX    = eval_state['n_evaluated'] if eval_state else 0
if START_IDX > 0:
    logger.log(f'Resuming evaluate from record {START_IDX}')
else:
    logger.log('Starting fresh evaluate')

[    56.6s] Test records: 701
[    56.6s] Starting fresh evaluate


## Inference Functions

In [13]:
ENTITY_TYPES = ['PER', 'LOC', 'ORG', 'DTM', 'TITLE']

def parse_entities(text):
    return set(re.findall(r'\{([^|]+)\|([^}]+)\}', text))

def make_prompt(record):
    return (
        f"### Instruction:\n{record['instruction']}\n\n"
        f"### Input:\n{record['input']}\n\n"
        f"### Output:\n"
    )

def generate_batch(records):
    prompts = [make_prompt(r) for r in records]
    inputs  = tokenizer(
        prompts,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = CONFIG['max_seq_len'],
        padding        = True,
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = CONFIG['max_new_tokens'],
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )
    return [
        tokenizer.decode(o[input_len:], skip_special_tokens=True).strip()
        for o in outputs
    ]

def compute_prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2*p*r / (p+r) if (p+r) > 0 else 0
    return p, r, f

logger.log('Functions ready.')

[    56.6s] Functions ready.


## Evaluate Loop (Resume-safe)

In [ ]:
# Load previous results nếu resume
if eval_state and START_IDX > 0:
    prev_path = f"{CONFIG['result_dir']}/eval_checkpoint_{START_IDX}.json"
    if os.path.exists(prev_path):
        with open(prev_path) as f:
            prev = json.load(f)
        overall          = prev['overall_counts']
        per_type         = prev['per_type_counts']
        predictions      = prev['predictions']
        all_gold_entities = [(e['surf'],e['type'],e['slen']) for e in prev.get('gold_entities',[])]
        missed_entities   = [(e['surf'],e['type'],e['slen']) for e in prev.get('missed_entities',[])]
        wrong_entities    = [(e['surf'],e['pred'],e['gold']) for e in prev.get('wrong_entities',[])]
        logger.log(f'Loaded {START_IDX} previous results.')
    else:
        START_IDX = 0
        logger.log('Checkpoint file not found, starting fresh.')

if START_IDX == 0:
    overall           = {'tp': 0, 'fp': 0, 'fn': 0}
    per_type          = {t: {'tp': 0, 'fp': 0, 'fn': 0} for t in ENTITY_TYPES}
    predictions       = []
    all_gold_entities = []
    missed_entities   = []
    wrong_entities    = []

BATCH_SIZE = CONFIG['eval_batch']
SAVE_EVERY = CONFIG['save_every']

remaining  = test_data[START_IDX:]
logger.log(f'Evaluating {len(remaining)} remaining records (total {len(test_data)})')

for batch_start in range(0, len(remaining), BATCH_SIZE):
    batch       = remaining[batch_start:batch_start + BATCH_SIZE]
    batch_preds = generate_batch(batch)

    for record, pred_text in zip(batch, batch_preds):
        sent_len  = len(record['input'])
        gold_ents = parse_entities(record['output'])
        pred_ents = parse_entities(pred_text)

        overall['tp'] += len(gold_ents & pred_ents)
        overall['fp'] += len(pred_ents - gold_ents)
        overall['fn'] += len(gold_ents - pred_ents)

        for etype in ENTITY_TYPES:
            g = {e for e in gold_ents if e[1] == etype}
            p = {e for e in pred_ents if e[1] == etype}
            per_type[etype]['tp'] += len(g & p)
            per_type[etype]['fp'] += len(p - g)
            per_type[etype]['fn'] += len(g - p)

        for surf, etype in gold_ents:
            all_gold_entities.append((surf, etype, sent_len))
        for surf, etype in (gold_ents - pred_ents):
            missed_entities.append((surf, etype, sent_len))
        gold_surf = {s: t for s, t in gold_ents}
        pred_surf = {s: t for s, t in pred_ents}
        for surf in set(gold_surf) & set(pred_surf):
            if gold_surf[surf] != pred_surf[surf]:
                wrong_entities.append((surf, pred_surf[surf], gold_surf[surf]))

        predictions.append({
            'input':     record['input'],
            'gold':      record['output'],
            'pred':      pred_text,
            'sent_len':  sent_len,
            'n_gold':    len(gold_ents),
            'n_pred':    len(pred_ents),
            'n_correct': len(gold_ents & pred_ents),
        })

    done    = START_IDX + min(batch_start + BATCH_SIZE, len(remaining))
    elapsed = time.time() - logger.start
    per_rec = elapsed / max(done - START_IDX, 1)
    eta     = per_rec * (len(test_data) - done)
    _, _, f1_now = compute_prf(**overall)

    logger.log(
        f'[{done:>3}/{len(test_data)}] '
        f'{per_rec:.1f}s/rec | '
        f'Elapsed: {elapsed/60:.1f}m | '
        f'ETA: {eta/60:.1f}m | '
        f'F1: {f1_now:.4f}'
    )

    # Auto-save mỗi SAVE_EVERY records
    if done % SAVE_EVERY == 0:
        ckpt_data = {
            'n_evaluated':    done,
            'overall_counts': overall,
            'per_type_counts':per_type,
            'predictions':    predictions,
            'gold_entities':  [{'surf':s,'type':t,'slen':l} for s,t,l in all_gold_entities],
            'missed_entities':[{'surf':s,'type':t,'slen':l} for s,t,l in missed_entities],
            'wrong_entities': [{'surf':s,'pred':p,'gold':g} for s,p,g in wrong_entities],
        }
        ckpt_path = f"{CONFIG['result_dir']}/eval_checkpoint_{done}.json"
        with open(ckpt_path, 'w', encoding='utf-8') as f:
            json.dump(ckpt_data, f, ensure_ascii=False)
        logger.save_state({'n_evaluated': done}, 'eval_state')
        logger.log(f'Checkpoint saved: {ckpt_path}')

logger.log(f'Evaluate done! Total: {(time.time()-logger.start)/60:.1f} min')

Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[    56.6s] Evaluating 701 remaining records (total 701)


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[    93.1s] [  4/701] 23.3s/rec | Elapsed: 1.6m | ETA: 270.4m | F1: 0.1553


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[  8082.2s] [692/701] 11.7s/rec | Elapsed: 134.7m | ETA: 1.8m | F1: 0.1003


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[  8115.5s] [696/701] 11.7s/rec | Elapsed: 135.3m | ETA: 1.0m | F1: 0.1006


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[  8148.8s] [700/701] 11.6s/rec | Elapsed: 135.8m | ETA: 0.2m | F1: 0.1010
[  8148.9s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/eval_state.json
[  8148.9s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/results/eval_checkpoint_700.json
[  8160.0s] [701/701] 11.6s/rec | Elapsed: 136.0m | ETA: 0.0m | F1: 0.1014
[  8160.0s] Evaluate done! Total: 136.0 min


## F1 Results

In [ ]:
print('=' * 55)
print(f"{'Entity':<10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 55)
for etype in ENTITY_TYPES:
    p, r, f = compute_prf(**per_type[etype])
    print(f'{etype:<10} {p:>10.4f} {r:>10.4f} {f:>10.4f}')
print('-' * 55)
p, r, f = compute_prf(**overall)
print(f"{'Overall':<10} {p:>10.4f} {r:>10.4f} {f:>10.4f}")
print('=' * 55)
print(f'\n>> Overall F1       : {f:.4f}')
print(f'>> Baseline (Paper 1): 0.8200')
print(f'>> Delta             : {f - 0.82:+.4f}')

Entity      Precision     Recall         F1
-------------------------------------------------------
PER            0.2430     0.1130     0.1543
LOC            0.3632     0.0705     0.1181
ORG            0.1644     0.0253     0.0438
DTM            0.2024     0.0278     0.0489
TITLE          0.1347     0.0534     0.0765
-------------------------------------------------------
Overall        0.2209     0.0658     0.1014

>> Overall F1       : 0.1014
>> Baseline (Paper 1): 0.8200
>> Delta             : -0.7186


## Error Analysis

In [ ]:
# 6.1 Miss rate per type
print('=== Miss Rate per Entity Type ===')
print(f"{'Type':<10} {'Gold':>8} {'Missed':>8} {'Miss%':>8}")
print('-' * 38)
for etype in ENTITY_TYPES:
    n_gold   = per_type[etype]['tp'] + per_type[etype]['fn']
    n_missed = per_type[etype]['fn']
    pct      = n_missed/n_gold*100 if n_gold > 0 else 0
    print(f'{etype:<10} {n_gold:>8} {n_missed:>8} {pct:>7.1f}%')

print()
# 6.2 Type confusion
print('=== Type Confusion ===')
confusion = Counter([(g, p) for _, p, g in wrong_entities])
for (gold, pred), count in confusion.most_common(10):
    print(f'{gold:<8} -> {pred:<8} {count:>6}')

print()
# 6.3 Top missed
print('=== Top 20 Missed Entities ===')
missed_counter = Counter([(s, t) for s, t, _ in missed_entities])
for (surf, etype), count in missed_counter.most_common(20):
    print(f'{surf:<20} {etype:<8} {count:>6}')

print()
# 6.4 Entity length
print('=== Entity Length vs Miss Rate ===')
def elen(s):
    n = len(s)
    if n==1: return '1'
    if n==2: return '2'
    if n<=4: return '3-4'
    return '5+'
gold_bl   = Counter(elen(s) for s,_,_ in all_gold_entities)
missed_bl = Counter(elen(s) for s,_,_ in missed_entities)
for b in ['1','2','3-4','5+']:
    g = gold_bl.get(b,0); m = missed_bl.get(b,0)
    print(f'{b:<6} Gold:{g:>6}  Missed:{m:>6}  {m/g*100 if g else 0:>5.1f}%')

print()
# 6.5 Sentence length
print('=== Sentence Length vs F1 ===')
def slen(l):
    if l<=50: return '<=50'
    if l<=100: return '51-100'
    if l<=150: return '101-150'
    if l<=200: return '151-200'
    return '200+'
bs = defaultdict(lambda: {'tp':0,'fp':0,'fn':0,'n':0})
for pred in predictions:
    b = slen(pred['sent_len'])
    bs[b]['tp'] += pred['n_correct']
    bs[b]['fp'] += pred['n_pred'] - pred['n_correct']
    bs[b]['fn'] += pred['n_gold'] - pred['n_correct']
    bs[b]['n']  += 1
for b in ['<=50','51-100','101-150','151-200','200+']:
    if b in bs:
        s = bs[b]; _,_,f = compute_prf(s['tp'],s['fp'],s['fn'])
        print(f'{b:<12} Sents:{s["n"]:>4}  F1:{f:.4f}')

=== Miss Rate per Entity Type ===
Type           Gold   Missed    Miss%
--------------------------------------
PER            2920     2590    88.7%
LOC            2354     2188    92.9%
ORG            1938     1889    97.5%
DTM            1222     1188    97.2%
TITLE          1947     1843    94.7%

=== Type Confusion ===
TITLE    -> PER          54
ORG      -> TITLE        27
PER      -> TITLE        22
ORG      -> PER          16
LOC      -> PER          12
LOC      -> TITLE        11
ORG      -> LOC          11
LOC      -> ORG           9
LOC      -> DTM           9
TITLE    -> ORG           5

=== Top 20 Missed Entities ===
LO                   LOC          19
"(                   PER          19
"(                   LOC          19
(P                   PER          17
PE                   PER          16
"                    ORG          15
.                    ORG          14
(L                   LOC          14
R)                   PER          13
IT                   TITLE    

In [ ]:
# Save final results
final_results = {
    'overall':  dict(zip(['precision','recall','f1'], compute_prf(**overall))),
    'per_type': {t: dict(zip(['precision','recall','f1'], compute_prf(**per_type[t]))) for t in ENTITY_TYPES},
    'n_evaluated': len(predictions),
    'predictions': predictions,
    'error_analysis': {
        'top_missed': [{'surface':s,'type':t,'count':c} for (s,t),c in missed_counter.most_common(50)],
        'type_confusion': [{'gold':g,'pred':p,'count':c} for (g,p),c in confusion.most_common()],
    }
}
out_path = f"{CONFIG['result_dir']}/eval_final.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)
logger.log(f'Final results saved: {out_path}')
print(f'F1 = {final_results["overall"]["f1"]:.4f}')

[  8160.0s] Final results saved: /content/drive/MyDrive/dvsktt_ner/results/eval_final.json
F1 = 0.1014
